In [1]:
!pip install pypdf langchain langchain-core langchain-community langchain-text-splitters chromadb sentence-transformers rank-bm25 langchain-groq -q

In [7]:
import numpy as np
import os
import json
import torch
import uuid
from pathlib import Path
from langchain_core.documents import Document

In [2]:
import chromadb

In [3]:
from chromadb.config import Settings

In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

C:\Users\Bikram\anaconda3\envs\earning_call-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

True

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [9]:
class EmbeddingManager:
    def __init__(self, model_name="BAAI/bge-small-en-v1.5", device="cpu"):
        self.model_name = model_name
        self.device = device
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name, device=self.device)
        print("embedding dimensions =", self.model.get_embedding_dimension())

    def generate_embeddings(self, texts):
        embeddings = self.model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
        return embeddings

In [10]:
print("Loading reranker model...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    device=device
)
print("Reranker model loaded")

Loading reranker model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3754.43it/s]


Reranker model loaded


In [11]:
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\Bikram\AppData\Local\Temp\ipykernel_25184\884480341.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [12]:
def extract_text_from_pdf(pdf_path):
    """
    Extract text using PyPDFLoader to align with the PDF class style.
    Returns list of Document objects.
    """
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    
    # Clean text formatting on each page
    for doc in docs:
        doc.page_content = " ".join(doc.page_content.split())
        
    print(f"Extracted {len(docs)} pages from PDF")
    return docs


In [13]:
def extract_text_from_txt(txt_path):
    """
    Fallback loader for plain text files.
    Splits content into artificial Document pages.
    """
    with open(txt_path, encoding="utf-8") as f:
        full_text = f.read()

    words = full_text.split()
    page_size = 1000
    docs = []

    for i in range(0, len(words), page_size):
        chunk = " ".join(words[i:i + page_size])
        docs.append(Document(
            page_content=chunk,
            metadata={"source": txt_path, "page": (i // page_size) + 1}
        ))

    print(f"Split into {len(docs)} sections")
    return docs

In [14]:
def split_docs(documents, chunk_size=512, chunk_overlap=64):
    """
    Splits Document objects using RecursiveCharacterTextSplitter.
    Converts chunked output into indexed chunk dictionaries with metadata.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
    )
    
    chunked_docs = text_splitter.split_documents(documents)
    processed_chunks = []
    
    for idx, doc in enumerate(chunked_docs):
        # Normalize and align 1-based page indices
        page_num = doc.metadata.get("page", 0) + 1 if "page" in doc.metadata else 1
        
        if len(doc.page_content.strip()) < 30:
            continue
            
        processed_chunks.append({
            "chunk_id": idx,
            "page_num": page_num,
            "text": doc.page_content.strip(),
            "word_count": len(doc.page_content.split())
        })

    print(f"Total chunks created: {len(processed_chunks)}")
    print(f"Avg words per chunk: {round(np.mean([c['word_count'] for c in processed_chunks]), 1)}")
    return processed_chunks

In [15]:
class VectorStoreManager:
    def __init__(self, collection_name="transcript"):
        self.collection_name = collection_name
        self.client = chromadb.Client(Settings(anonymized_telemetry=False))
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            pass

        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    def add_chunks(self, chunks, embeddings):
        ids = [f"chunk_{c['chunk_id']}" for c in chunks]
        documents_content = [c["text"] for c in chunks]
        all_metadata = [
            {
                "chunk_id": c["chunk_id"],
                "page_num": c["page_num"],
                "word_count": c["word_count"]
            }
            for c in chunks
        ]

        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            self.collection.add(
                ids=ids[i:i + batch_size],
                embeddings=embeddings[i:i + batch_size].tolist(),
                documents=documents_content[i:i + batch_size],
                metadatas=all_metadata[i:i + batch_size]
            )
        print(f"Indexed {self.collection.count()} chunks in ChromaDB")

In [16]:
def build_bm25_index(chunks):
    """
    Build BM25 keyword index over all chunks.
    Used for hybrid retrieval alongside semantic search.
    """
    tokenized_corpus = [c["text"].lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    print("BM25 index built")
    return bm25

In [17]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store, bm25, chunks):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
        self.bm25 = bm25
        self.chunks = chunks

    def retrieve(self, query, top_k=20, alpha=0.6):
        """
        Retrieves top_k elements blending Semantic Search and BM25.
        """
        # --- Semantic Retrieval ---
        query_embedding = self.embedding_manager.model.encode(
            query,
            normalize_embeddings=True,
            prompt="Represent this sentence for searching relevant passages: "
        )

        semantic_results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        semantic_scores = {}
        if semantic_results["ids"] and semantic_results["distances"]:
            for chunk_id_str, distance in zip(
                semantic_results["ids"][0],
                semantic_results["distances"][0]
            ):
                chunk_id = int(chunk_id_str.replace("chunk_", ""))
                semantic_scores[chunk_id] = 1 - distance

        # --- BM25 Retrieval ---
        tokenized_query = query.lower().split()
        bm25_scores_raw = self.bm25.get_scores(tokenized_query)

        max_bm25 = max(bm25_scores_raw) if max(bm25_scores_raw) > 0 else 1.0
        bm25_scores_normalized = bm25_scores_raw / max_bm25

        top_bm25_indices = np.argsort(bm25_scores_normalized)[::-1][:top_k]
        bm25_scores = {idx: bm25_scores_normalized[idx] for idx in top_bm25_indices}

        # --- Fusion Scorer ---
        all_chunk_ids = set(semantic_scores.keys()) | set(bm25_scores.keys())
        combined = []
        for chunk_id in all_chunk_ids:
            sem_score = semantic_scores.get(chunk_id, 0.0)
            bm25_score = bm25_scores.get(chunk_id, 0.0)
            combined_score = alpha * sem_score + (1 - alpha) * bm25_score

            combined.append({
                **self.chunks[chunk_id],
                "semantic_score": round(sem_score, 4),
                "bm25_score":     round(float(bm25_score), 4),
                "combined_score": round(combined_score, 4)
            })

        combined.sort(key=lambda x: x["combined_score"], reverse=True)
        return combined[:top_k]

In [1]:
def rerank_chunks(query, candidates, top_n = 5):
    if not candidates:
        return []

    pairs = [(query, c["text"]) for c in candidates]
    
    # 1. Score all pairs
    rerank_scores = reranker.predict(pairs)

    valid_candidates = []
    for i, candidate in enumerate(candidates):
        score = float(rerank_scores[i])
        
        # 2. ONLY keep the text if the score is greater than 0
        if score > 0.0:
            candidate["rerank_score"] = score
            valid_candidates.append(candidate)

    # 3. Sort only the good chunks
    reranked = sorted(valid_candidates, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_n]

In [18]:
from langchain_groq import ChatGroq

In [39]:
def generate_answer(query, retrieved_chunks):
    if not retrieved_chunks:
        return {
            "answer": "I could not find relevant information in the transcript to answer this question.",
            "sources": []
        }

    # Context construction from retrieved resources
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks):
        context_parts.append(
            f"[Source {i+1} — Page {chunk['page_num']}]\n{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    system_prompt = """You are a precise financial analyst assistant analyzing earnings call transcripts.

Your job is to answer questions about the transcript using ONLY the provided context.

Rules:
- Answer directly and specifically from the context
- If the answer contains specific numbers, metrics, or quotes — include them exactly
- Answer ONLY using the retrieved context.
- If the information is absent, explicitly say:
   "The uploaded transcript does not contain this information."
- Never infer financial facts.
- Always mention which source/page you found the answer in
- If the context does not contain enough information to answer — say so clearly
- Never make up information not present in the context
- Never fabricate speaker statements.
- Keep answers concise but complete"""

    user_message = f"Question: {query}\n\nContext from transcript:\n{context}\n\nAnswer the question based only on the context above."

    try:
        # Initialize Groq LLM in LangChain wrapper style
        llm = ChatGroq(
            groq_api_key=os.getenv("GROQ_API_KEY"),
            model="llama-3.3-70b-versatile",
            temperature=0.1,
            max_tokens=600
        )

        response = llm.invoke([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ])

        answer = response.content.strip()

        # Build source output details
        sources = [
            {
                "page_num":     chunk["page_num"],
                "chunk_id":     chunk["chunk_id"],
                "text_preview": chunk["text"][:200],
                "rerank_score": round(chunk["rerank_score"], 4)
            }
            for chunk in retrieved_chunks
        ]

        return {
            "answer":  answer,
            "sources": sources
        }

    except Exception as e:
        return {
            "answer":  f"Error generating answer: {e}",
            "sources": []
        }

In [40]:
def build_rag(file_path):
    """
    Build the complete RAG pipeline once for a document.
    Returns a retriever object that can answer multiple queries.
    """

    # Step 1: Document Parsing
    if file_path.endswith(".pdf"):
        documents = extract_text_from_pdf(file_path)
    else:
        documents = extract_text_from_txt(file_path)

    # Step 2: Split Documents
    chunks = split_docs(
        documents,
        chunk_size=512,
        chunk_overlap=64
    )

    # Step 3: Generate Embeddings
    embedding_manager = EmbeddingManager(
        model_name="BAAI/bge-small-en-v1.5",
        device=device
    )

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_manager.generate_embeddings(texts)

    # Step 4: Build Vector Store
    vector_store = VectorStoreManager(
        collection_name="transcript"
    )

    vector_store.add_chunks(
        chunks,
        embeddings
    )

    # Step 5: Build BM25
    bm25 = build_bm25_index(chunks)

    # Step 6: Create Retriever

    retriever = RAGRetriever(
        embedding_manager=embedding_manager,
        vector_store=vector_store,
        bm25=bm25,
        chunks=chunks
    )

    return retriever

In [41]:
def ask_question(
    retriever,
    query,
    top_k_retrieve=20,
    top_n_rerank=5,
    alpha=0.6
):
    """
    Ask multiple questions on an already-built RAG pipeline.
    """

    # Retrieval
    candidates = retriever.retrieve(
        query=query,
        top_k=top_k_retrieve,
        alpha=alpha
    )

    # Rerank
    final_chunks = rerank_chunks(
        query,
        candidates,
        top_n=top_n_rerank
    )

    # Generate Answer
    result = generate_answer(
        query,
        final_chunks
    )

    return {
        "query": query,
        "answer": result["answer"],
        "sources": result["sources"],
        "chunks_retrieved": len(candidates),
        "chunks_reranked": len(final_chunks)
    }

In [37]:
test_txt = "../data/raw_transcripts/AAPL_Q1_2023.json"

# Convert transcript payload for text pipeline testing
with open(test_txt, encoding="utf-8") as f:
    raw = json.load(f)

temp_path = "temp_test_transcript.txt"
with open(temp_path, "w", encoding="utf-8") as f:
    f.write(raw["content"])
retriever = build_rag(temp_path)

Split into 8 sections
Total chunks created: 110
Avg words per chunk: 73.4
loading model.... BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3811.61it/s]


embedding dimensions = 384


Batches: 100%|██████████| 4/4 [00:00<00:00, 11.92it/s]

Indexed 110 chunks in ChromaDB
BM25 index built


In [38]:
def ask_anything(test_queries):
    for query in test_queries:
        print("=" * 60)
        print(f"QUERY: {query}")
        print()
    
        result = ask_question(
            retriever,
            query
        )
    
        print(f"ANSWER:\n{result['answer']}")
        print()
        print(f"SOURCES:")
        for src in result["sources"]:
            print(f"  Page {src['page_num']} (score: {src['rerank_score']}): {src['text_preview'][:200]}...")
        print()